In [1]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

In [2]:
n_data_list = [1_000, 1_000_000]
dim_list = [2, 5]
metrics = ['euclidean', 'cosine']
k = 10

X_list=[]
query_list = []
skenario_info = []

np.random.seed(42)

In [3]:
for n_data in n_data_list:
    for dim in dim_list:
        X = np.random.random((n_data, dim)).astype(np.float32)
        query = np.random.random((1, dim)).astype(np.float32)
        X_list.append(X)
        query_list.append(query)        
        skenario_info.append((n_data, dim))

In [8]:
from scipy.ndimage import label
for (n_data,dim), X_raw, query_raw in zip(skenario_info, X_list, query_list):
    for metric in metrics:
        X=X_raw.copy()
        query=query_raw.copy()
        if metric == 'cosine':
            faiss.normalize_L2(X)
            faiss.normalize_L2(query)
        
        # ANNOY
        ann_metric='euclidean' if metric == 'euclidean' else 'angular'
        ann_index=AnnoyIndex(dim, ann_metric)
        start = time.time()
        for i in range(n_data):
            ann_index.add_item(i,X[i])
        ann_index.build(10)
        ann_build = time.time() - start

        start = time.time()
        ann_neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
        ann_query = time.time() - start

        # FAISS
        faiss_index = faiss.IndexFlatL2(dim) if metric == 'euclidean' else faiss.IndexFlatIP(dim)
        start = time.time()
        faiss_index.add(X)
        faiss_build = time.time()-start

        start = time.time()
        _, faiss_indices = faiss_index.search(query, k)
        faiss_query = time.time() - start

        # HNSW
        hnsw_metric = 'l2' if metric=='euclidean' else 'cosine'
        hnsw_index = hnswlib.Index(space=hnsw_metric, dim=dim)
        start = time.time()
        hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
        hnsw_index.add_items(X)
        hnsw_build = time.time()-start
        
        hnsw_index.set_ef(50)
        start = time.time()
        hnsw_labels, _ = hnsw_index.knn_query(query, k=k)
        hnsw_query = time.time()-start

        print(f"\nData: {n_data:,} | Dimensi: {dim}D | Metrik: {metric.upper()}")
        print(f'- ANNOY:\nBuild time: {ann_build}s \nQuery time: {ann_query}s \nNeighbors: {ann_neighbors[0][:5]}')
        print(f'- FAIZZ:\nBuild time: {faiss_build}s \nQuery time: {faiss_query}s \nNeighbors: {faiss_indices[0][:5]}')
        print(f'- HNSW:\nBuild time: {hnsw_build}s \nQuery time: {hnsw_query}s \nNeighbors: {hnsw_labels[0][:5]}')


Data: 1,000 | Dimensi: 2D | Metrik: EUCLIDEAN
- ANNOY:
Build time: 0.013415813446044922s 
Query time: 0.000110626220703125s 
Neighbors: [112, 535, 777, 246, 763]
- FAIZZ:
Build time: 6.842613220214844e-05s 
Query time: 0.0006537437438964844s 
Neighbors: [112 535 777 246 763]
- HNSW:
Build time: 0.02188730239868164s 
Query time: 8.058547973632812e-05s 
Neighbors: [112 535 777 246 763]

Data: 1,000 | Dimensi: 2D | Metrik: COSINE
- ANNOY:
Build time: 0.014997482299804688s 
Query time: 8.344650268554688e-05s 
Neighbors: [738, 183, 561, 91, 244]
- FAIZZ:
Build time: 3.600120544433594e-05s 
Query time: 4.220008850097656e-05s 
Neighbors: [738 183 561  91 244]
- HNSW:
Build time: 0.014060497283935547s 
Query time: 5.7697296142578125e-05s 
Neighbors: [738 183 561  91 244]

Data: 1,000 | Dimensi: 5D | Metrik: EUCLIDEAN
- ANNOY:
Build time: 0.007747650146484375s 
Query time: 9.822845458984375e-05s 
Neighbors: [363, 660, 764, 482, 185]
- FAIZZ:
Build time: 0.000110626220703125s 
Query time: 6.914

### Benchmark Performa Algoritma Vector Search (ANNOY vs FAISS vs HNSW)

| Jumlah Data | Dimensi | Metrik | Algoritma | Build Time (s) | Query Time (s) | Neighbors |
|:---:|:---:|:---:|:---:|---:|---:|:---|
| 1,000 | 2D | EUCLIDEAN | ANNOY | 0.013416 | 0.000111 | `[112, 535, 777, 246, 763]` |
| 1,000 | 2D | EUCLIDEAN | FAISS | 0.000068 | 0.000654 | `[112, 535, 777, 246, 763]` |
| 1,000 | 2D | EUCLIDEAN | HNSW | 0.021887 | 0.000081 | `[112, 535, 777, 246, 763]` |
| 1,000 | 2D | COSINE | ANNOY | 0.014997 | 0.000083 | `[738, 183, 561, 91, 244]` |
| 1,000 | 2D | COSINE | FAISS | 0.000036 | 0.000042 | `[738, 183, 561, 91, 244]` |
| 1,000 | 2D | COSINE | HNSW | 0.014060 | 0.000058 | `[738, 183, 561, 91, 244]` |
| 1,000 | 5D | EUCLIDEAN | ANNOY | 0.007748 | 0.000098 | `[363, 660, 764, 482, 185]` |
| 1,000 | 5D | EUCLIDEAN | FAISS | 0.000111 | 0.000069 | `[363, 660, 764, 482, 185]` |
| 1,000 | 5D | EUCLIDEAN | HNSW | 0.015674 | 0.000080 | `[363, 660, 764, 482, 185]` |
| 1,000 | 5D | COSINE | ANNOY | 0.008159 | 0.000069 | `[185, 168, 660, 363, 764]` |
| 1,000 | 5D | COSINE | FAISS | 0.000028 | 0.000036 | `[185, 168, 660, 363, 764]` |
| 1,000 | 5D | COSINE | HNSW | 0.013147 | 0.000061 | `[185, 168, 660, 363, 764]` |
| 1,000,000 | 2D | EUCLIDEAN | ANNOY | 6.988580 | 0.000131 | `[654158, 879917, 561958, 639324, 998926]` |
| 1,000,000 | 2D | EUCLIDEAN | FAISS | 0.002540 | 0.006967 | `[654158, 879917, 561958, 639324, 998926]` |
| 1,000,000 | 2D | EUCLIDEAN | HNSW | 15.931354 | 0.000132 | `[654158, 879917, 561958, 639324, 998926]` |
| 1,000,000 | 2D | COSINE | ANNOY | 15.291960 | 0.000133 | `[93286, 181892, 211929, 352586, 450840]` |
| 1,000,000 | 2D | COSINE | FAISS | 0.002875 | 0.001725 | `[918941, 821785, 635824, 622988, 501434]` |
| 1,000,000 | 2D | COSINE | HNSW | 24.283714 | 0.000198 | `[37271, 222102, 262660, 489069, 521675]` |
| 1,000,000 | 5D | EUCLIDEAN | ANNOY | 6.578249 | 0.000172 | `[112836, 698526, 151916, 220675, 883190]` |
| 1,000,000 | 5D | EUCLIDEAN | FAISS | 0.009571 | 0.004800 | `[112836, 698526, 151916, 220675, 883190]` |
| 1,000,000 | 5D | EUCLIDEAN | HNSW | 24.216948 | 0.000184 | `[112836, 698526, 151916, 220675, 883190]` |
| 1,000,000 | 5D | COSINE | ANNOY | 8.122095 | 0.000162 | `[610480, 80626, 814735, 980891, 698974]` |
| 1,000,000 | 5D | COSINE | FAISS | 0.006844 | 0.002540 | `[610480, 80626, 814735, 980891, 698974]` |
| 1,000,000 | 5D | COSINE | HNSW | 23.637605 | 0.000330 | `[610480, 80626, 814735, 980891, 698974]` |